# 01-02 — Construction des données et découpage temporel

**Objectif :** appliquer le feature engineering décidé dans `01_01_exploration_donnees` et produire des ensembles propres, **sans fuite de données**, prêts pour la modélisation.

Chaque ligne = **un équipement à une date de coupure** (dernier jour du mois), avec ses variables (son passé, déjà saisi à cette date) et sa cible de survie (délai jusqu'à la prochaine panne, `duree`, et panne observée ou non, `evenement`).

| Ensemble | Dates de coupure | Fin d'observation | Rôle |
|---|---|---|---|
| Entraînement | fin janvier 2019 → fin octobre 2024 | 30/11/2024 | le modèle apprend |
| Validation | fin novembre 2024 → fin avril 2025 | 31/05/2025 | choix des variables et des réglages |
| Test | fin août 2025 → fin novembre 2025 | 31/12/2025 | évaluation finale, utilisé une seule fois |
| Futur | 31/12/2025 | — | prévision réelle, sans cible |

| Paramètre | Valeur | Justification |
|---|---|---|
| Découpage | **temporel** (pas aléatoire) | on prédit toujours l'avenir à partir du passé |
| Suivi maximal `H_MAX` | 365 jours | horizons utiles : de quelques jours à un an |
| Trou d'extraction | juin-juillet 2025 | aucune date de coupure, aucun suivi dans ce trou |
| `RANDOM_STATE` | 42 | reproductibilité des contrôles par échantillon |

> **Gardes anti-fuite :**
> 1. les variables d'une date de coupure n'utilisent que les OT **réalisés et déjà saisis** ce jour-là ;
> 2. la cible d'un ensemble ne regarde **jamais** les pannes après sa fin d'observation : l'entraînement ne voit pas la période de validation, la validation ne voit pas celle du test.

## 0. Imports globaux

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from settings.params import (
    DATA_PROCESSED, RANDOM_STATE, H_MAX, HORIZONS, TYPE_PANNE, TROU_DEBUT, TROU_FIN,
    COUPURES_TRAIN, FIN_OBS_TRAIN, COUPURES_VAL, FIN_OBS_VAL, COUPURES_TEST, FIN_OBS_TEST, DATE_FUTUR,
    FEATURES_NUM, FEATURES_CAT, FEATURES_ALL, FEATURES_LINEAR, DUREE, EVENEMENT,
)
from src.data.make_dataset import load_data
from src.data.preprocess import prepare
from src.features.build import EQ, construire_ensembles, ot_visibles, kaplan_meier, proba_panne
from src.utils.logger import logger

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Chargement des données brutes

### Description

`load_data` lit l'export GMAO `data/raw/FaitOT.xlsx`. Les données brutes ne sont jamais modifiées : toutes les transformations sont faites sur des copies.

In [ ]:
data = load_data()
data.head(3)

## 2. Préparation des OT

`prepare` (fichier `src/data/preprocess.py`) applique les décisions de l'exploration :

| Étape | Rôle |
|---|---|
| Précautions de format | type d'intervention en majuscules, codes en texte, espaces parasites retirés |
| Doublons de `codeOT` | une ligne par OT (la ligne avec l'unité plutôt que `DD`) |
| Dates | `date_realisation` = jour de l'intervention (date de la panne) ; `date_connue` = jour où l'OT devient visible = le plus tardif entre réalisation et saisie |
| Période | OT réalisés du 1er janvier 2019 au 31 décembre 2025 |

In [ ]:
ot = prepare(data)
print(f"OT réalisés avant d'être connus (saisis plus tard) : {(ot['date_connue'] > ot['date_realisation']).mean():.1%}")
ot[[EQ, "typeIntervention", "date_realisation", "date_connue", "familleEquipement", "zone", "fonction"]].head(3)

## 3. Variables et cible, ensemble par ensemble

`construire_ensembles` (fichier `src/features/build.py`) fait, pour chaque ensemble :
1. **les dates de coupure** de l'ensemble (dernier jour de chaque mois, sans le trou d'extraction) ;
2. **les variables** de chaque équipement connu à chaque date (`variables_a_date`), avec les OT déjà saisis ;
3. **la cible** (`ajouter_cible`) : jours jusqu'à la prochaine panne réalisée, en arrêtant le suivi au plus tôt de : 365 jours, fin d'observation de l'ensemble, veille du trou d'extraction ;
4. **la colonne `hors_perimetre`** : localités, unités, groupes d'équipements et véhicules, à exclure de l'évaluation et de la liste de risque.

Le **futur** (31 décembre 2025) n'a que les variables : c'est sur lui que le modèle final fera la prévision réelle.

In [ ]:
ensembles = construire_ensembles(ot)
tr, va, te, futur = ensembles["train"], ensembles["val"], ensembles["test"], ensembles["futur"]
tr[[EQ, "date_coupure"] + FEATURES_NUM[:6] + ["familleEquipement", "hors_perimetre", DUREE, EVENEMENT]].head(5)

## 4. Contrôles anti-fuite

Cinq vérifications automatiques. Si l'une échoue, le notebook s'arrête avec une erreur (`assert`).

1. **Ordre des dates** : toutes les dates de coupure de l'entraînement précèdent celles de la validation, qui précèdent celles du test ; aucune date dans le trou d'extraction.
2. **Fin d'observation** : pour chaque ligne, date de coupure + `duree` ne dépasse jamais la fin d'observation de son ensemble.
3. **Trou d'extraction** : aucun suivi ne traverse juin-juillet 2025.
4. **Cible cohérente** : `duree` entre 1 et 365 jours ; quand `evenement = 1`, une panne existe bien ce jour-là.
5. **Variables sans futur** : sur 300 lignes tirées au hasard, on recalcule à la main le nombre de pannes sur 365 jours et les jours depuis la dernière panne avec les seuls OT réalisés et saisis à la date de coupure, et on compare.

In [ ]:
ens = {"train": (tr, FIN_OBS_TRAIN), "val": (va, FIN_OBS_VAL), "test": (te, FIN_OBS_TEST)}

# 1. Ordre des dates et trou d'extraction
assert tr["date_coupure"].max() < va["date_coupure"].min() < te["date_coupure"].min()
for nom, (d, _) in ens.items():
    assert not d["date_coupure"].between(TROU_DEBUT, TROU_FIN).any(), nom
print("1. Ordre des dates de coupure : OK")

# 2. Fin d'observation et 3. trou d'extraction
for nom, (d, fin_obs) in ens.items():
    fin_suivi = d["date_coupure"] + pd.to_timedelta(d[DUREE], unit="D")
    assert (fin_suivi <= pd.Timestamp(fin_obs)).all(), nom
    avant_trou = d["date_coupure"] < TROU_DEBUT
    assert (fin_suivi[avant_trou] < TROU_DEBUT).all(), nom
print("2. Aucune cible au-delà de la fin d'observation : OK")
print("3. Aucun suivi à travers le trou d'extraction : OK")

# 4. Cible cohérente
pannes = set(zip(ot.loc[ot["typeIntervention"] == TYPE_PANNE, EQ], ot.loc[ot["typeIntervention"] == TYPE_PANNE, "date_realisation"]))
for nom, (d, _) in ens.items():
    assert d[DUREE].between(1, H_MAX).all(), nom
    ev = d[d[EVENEMENT] == 1].sample(min(300, int(d[EVENEMENT].sum())), random_state=RANDOM_STATE)
    dates_panne = ev["date_coupure"] + pd.to_timedelta(ev[DUREE], unit="D")
    assert all((e, p) in pannes for e, p in zip(ev[EQ], dates_panne)), nom
print("4. Cible cohérente (durées entre 1 et 365 jours, pannes retrouvées) : OK")

# 5. Variables sans futur (recalcul à la main sur un échantillon)
ech = tr.sample(300, random_state=RANDOM_STATE)
erreurs = 0
for _, r in ech.iterrows():
    v = ot_visibles(ot[ot[EQ] == r[EQ]], r["date_coupure"])
    c = v[v["typeIntervention"] == TYPE_PANNE]["date_realisation"]
    n365 = (c > r["date_coupure"] - pd.Timedelta(days=365)).sum()
    jd = (r["date_coupure"] - c.max()).days if len(c) else 3000
    erreurs += (n365 != r["n_corr_365j"]) + (jd != r["jours_depuis_corr"])
assert erreurs == 0
print("5. Variables recalculées à la main sur 300 lignes : identiques")

### Analyse & Interprétation


Les **cinq contrôles passent** :
- les ensembles sont **dans l'ordre du temps**, et aucune date de coupure ne tombe dans le trou d'extraction ;
- **aucune cible ne regarde au-delà de la fin d'observation** de son ensemble : l'entraînement ignore tout ce qui se passe après le 30 novembre 2024, la validation tout ce qui se passe après le 31 mai 2025 ;
- **aucun suivi ne traverse** juin-juillet 2025 ;
- les durées sont toutes entre 1 et 365 jours, et chaque panne annoncée par la cible existe bien dans les OT ;
- sur 300 lignes prises au hasard, les variables recalculées à la main avec les seuls OT **réalisés et saisis** à la date de coupure sont **identiques** à celles du tableau.

C'est la garantie que le modèle sera entraîné et évalué **dans les conditions réelles** : il ne disposera jamais d'une information qui n'existait pas encore au moment de la prédiction.

## 5. Statistiques des ensembles

In [ ]:
lignes = []
for nom, d in [("train", tr), ("val", va), ("test", te), ("futur", futur)]:
    reel = d[~d["hors_perimetre"]]
    ligne = {"ensemble": nom, "lignes": len(d), "dates de coupure": d["date_coupure"].nunique(),
             "première": d["date_coupure"].min().date(), "dernière": d["date_coupure"].max().date(),
             "équipements": d[EQ].nunique(), "% hors périmètre": f"{d['hors_perimetre'].mean():.1%}"}
    if nom != "futur":
        km = kaplan_meier(reel[DUREE], reel[EVENEMENT])
        ligne.update({"suivi max (jours)": int(d[DUREE].max()), "% panne observée": f"{reel[EVENEMENT].mean():.1%}",
                      **{f"P(panne ≤ {h} j)": f"{proba_panne(km, h):.2%}" for h in HORIZONS}})
    lignes.append(ligne)
pd.DataFrame(lignes).set_index("ensemble")

### Analyse & Interprétation


- **Entraînement** : 957 924 lignes (70 dates de coupure, 32 999 équipements). **Validation** : 215 683 lignes (6 dates). **Test** : 168 425 lignes (4 dates). **Futur** : 43 805 équipements au 31 décembre 2025.
- **Codes hors périmètre** : leur part augmente avec le temps (17 % en entraînement, 27 % en test), car de nombreux codes `GRPMT` et localités ont été créés récemment. Ils restent dans les fichiers (colonne `hors_perimetre`), et seront retirés au moment de l'évaluation et de la liste de risque.
- **Suivi maximal** : 365 jours en entraînement, mais seulement 182 jours en validation et 122 jours en test, car ces périodes sont proches de la fin des données. C'est pourquoi la « part de pannes observées » est plus faible en validation (8,9 %) et en test (6,9 %) qu'en entraînement (21,7 %) : ce n'est pas un changement du risque, juste un suivi plus court.
- **Risque comparable d'un ensemble à l'autre** (Kaplan-Meier, équipements du réseau) : environ **1 %** de panne à 7 jours, **3,2 à 3,8 %** à 30 jours, **8 à 9 %** à 90 jours. Le risque est un peu plus faible en 2025 qu'en entraînement, ce qu'il faudra garder en tête au moment de l'évaluation.

**Conséquence pour l'évaluation :** sur le test, on pourra évaluer jusqu'à 90 jours environ (le suivi atteint 122 jours pour la date d'août et 92 jours pour celle de septembre). Les horizons plus longs (180 ou 365 jours) seront évalués sur la validation ou par backtest.

### Vérification des variables attendues

In [ ]:
for nom, d in [("train", tr), ("val", va), ("test", te), ("futur", futur)]:
    manquantes = set(FEATURES_ALL) - set(d.columns)
    if manquantes:
        raise ValueError(f"Variables manquantes dans {nom} : {manquantes}")
    assert d[FEATURES_NUM].notna().all().all(), f"valeur manquante dans une variable numérique ({nom})"
print("Toutes les variables sont présentes, aucune valeur manquante dans les variables numériques.")
print("\nCatégories inconnues (part des lignes) :")
display(pd.DataFrame({nom: d[FEATURES_CAT].isna().mean().round(4) for nom, d in [("train", tr), ("val", va), ("test", te), ("futur", futur)]}))
print(f"\nFEATURES_ALL    ({len(FEATURES_ALL)}) : {FEATURES_ALL}")
print(f"\nFEATURES_LINEAR ({len(FEATURES_LINEAR)}) : {FEATURES_LINEAR}")

### Analyse & Interprétation


- Les **17 variables** de `FEATURES_ALL` sont présentes dans les quatre fichiers, sans aucune valeur manquante dans les variables numériques (les cas « jamais » valent 0 ou 3 000, comme décidé dans l'exploration).
- **Catégories inconnues** : la famille manque pour 1,5 % des lignes d'entraînement et moins de 1 % ensuite ; la zone et la fonction sont presque toujours connues. Les modèles à arbres gèrent ces valeurs manquantes directement ; pour les modèles linéaires, elles deviendront la catégorie « INCONNU ».
- `FEATURES_LINEAR` (12 variables) est la liste sans variables redondantes, décidée en phase 7 de l'exploration.

## 6. Sauvegarde des ensembles

Format **Parquet** : sans perte, il conserve les types (dates, catégories) et se relit très vite. Le dossier `data/processed/` est créé si besoin ; il est exclu de Git (`.gitignore`), car il contient des données Senelec.

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
for nom, d in [("train", tr), ("val", va), ("test", te), ("futur", futur)]:
    chemin = DATA_PROCESSED / f"{nom}.parquet"
    d.to_parquet(chemin, index=False)
    logger.info(f"Sauvegardé {nom} -> {chemin.name} ({len(d):,} lignes, {chemin.stat().st_size / 1e6:.1f} Mo)")

relu = pd.read_parquet(DATA_PROCESSED / "train.parquet")
assert relu.shape == tr.shape and (relu.dtypes == tr.dtypes).all()
print("Relecture de train.parquet : identique (mêmes lignes, mêmes colonnes, mêmes types)")

## 7. Récapitulatif

| Fichier | Contenu | Usage |
|---|---|---|
| `data/processed/train.parquet` | dates de coupure de janvier 2019 à octobre 2024, avec cible | entraînement des modèles |
| `data/processed/val.parquet` | dates de coupure de novembre 2024 à avril 2025, avec cible | choix des variables et des réglages |
| `data/processed/test.parquet` | dates de coupure d'août à novembre 2025, avec cible | évaluation finale (utilisé une seule fois) |
| `data/processed/futur.parquet` | situation au 31 décembre 2025, sans cible | prévision réelle (API) |

Colonnes de chaque fichier : `codeEquipement`, `date_coupure`, les 17 variables (`FEATURES_ALL`), `hors_perimetre`, et pour les trois premiers la cible `duree` et `evenement`.

**Prochain notebook :** `02_modeling.ipynb`. Il comparera des **références simples** (taux moyen, nombre de pannes récentes) et des **modèles de survie** (Cox avec `FEATURES_LINEAR`, modèles à arbres avec `FEATURES_ALL`), évalués à 7, 30 et 90 jours, puis sauvegardera le modèle retenu pour l'API.